# 03 - Data Cleaning

## Objective

Clean and standardise the raw e-commerce dataset by addressing identified data quality issues, including missing values, duplicate records, inconsistent categorical values, invalid numerical values, and date-related issues.

The cleaned dataset will be validated and saved separately while preserving the original raw dataset.

## Source Dataset

`Dataset/Raw/ecommerce_sales_customer.csv`

## Output Dataset

`Dataset/Cleaned/ecommerce_sales_customer_cleaned.csv`

In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv(
    "../Dataset/Raw/ecommerce_sales_customer.csv"
)

In [3]:
df_clean = df.copy()

In [4]:
print(f"Original dataset shape: {df.shape}")
print(f"Working dataset shape: {df_clean.shape}")

Original dataset shape: (50000, 18)
Working dataset shape: (50000, 18)


In [5]:
df_clean.dtypes

OrderID                      object
OrderDate                    object
CustomerID                   object
CustomerRegistrationDate     object
ProductID                    object
ProductName                  object
Category                     object
Quantity                      int64
UnitPrice                   float64
DiscountPct                 float64
GrossSales                  float64
DiscountAmount              float64
NetSales                    float64
Region                       object
Country                      object
PaymentMethod                object
ReturnStatus                 object
ReturnDate                   object
dtype: object

In [6]:
df_clean.isnull().sum()

OrderID                         0
OrderDate                       0
CustomerID                      0
CustomerRegistrationDate        0
ProductID                       0
ProductName                     0
Category                       90
Quantity                        0
UnitPrice                       0
DiscountPct                     0
GrossSales                      0
DiscountAmount                  0
NetSales                        0
Region                         80
Country                         0
PaymentMethod                  79
ReturnStatus                    0
ReturnDate                  46810
dtype: int64

In [7]:
print(f"Duplicate records: {df_clean.duplicated().sum()}")

Duplicate records: 100


## 1. Duplicate Removal

Identify and remove exact duplicate records from the working dataset. The original raw dataset will remain unchanged.

In [8]:
initial_rows = len(df_clean)

print(f"Rows before duplicate removal: {initial_rows}")

Rows before duplicate removal: 50000


In [9]:
df_clean = df_clean.drop_duplicates()

In [10]:
print(
    "Maximum categories per product:",
    df_clean.groupby("ProductID")["Category"].nunique().max()
)

print(
    "Maximum regions per country:",
    df_clean.groupby("Country")["Region"].nunique().max()
)

print(
    "Remaining missing values:",
    df_clean.isnull().sum().sum()
)

Maximum categories per product: 4
Maximum regions per country: 1
Remaining missing values: 46966


In [11]:
print(f"Rows after duplicate removal: {len(df_clean)}")
print(f"Rows removed: {initial_rows - len(df_clean)}")
print(f"Remaining duplicates: {df_clean.duplicated().sum()}")

Rows after duplicate removal: 49900
Rows removed: 100
Remaining duplicates: 0


## 2. Missing Value Treatment

Investigate and handle missing values using related information where reliable. Preserve expected missing values and avoid introducing unsupported assumptions.

In [12]:
df_clean[df_clean["Category"].isna()][
    ["ProductID", "ProductName", "Category"]
].head(10)

,ProductID,ProductName,Category
927,P003,Smart Watch,NaN
2447,P008,Running Shoes,NaN
2896,P017,Skincare Bundle,NaN
3953,P007,Travel Backpack,NaN
4247,P012,Kitchen Organizer,NaN
4550,P012,Kitchen Organizer,NaN
4862,P011,Ceramic Mug Set,NaN
5452,P006,Laptop Sleeve,NaN
5806,P013,Desk Lamp,NaN
5997,P014,Yoga Mat,NaN


In [13]:
df_clean.groupby("ProductID")["Category"].nunique()

ProductID
P001    3
P002    4
P003    4
P004    4
P005    3
P006    4
P007    2
P008    4
P009    4
P010    3
P011    3
P012    4
P013    4
P014    3
P015    3
P016    4
P017    4
P018    3
P019    4
P020    4
Name: Category, dtype: int64

In [14]:
df_clean[df_clean["Region"].isna()][
    ["Country", "Region"]
].head(10)

,Country,Region
114,Pakistan,NaN
880,Australia,NaN
1187,Mexico,NaN
4379,South Africa,NaN
6507,Mexico,NaN
7162,Singapore,NaN
7457,Singapore,NaN
7465,France,NaN
7601,United Kingdom,NaN
8731,United Arab Emirates,NaN


In [15]:
df_clean.groupby("Country")["Region"].nunique()

Country
Australia               1
Brazil                  1
Canada                  1
Chile                   1
France                  1
Germany                 1
India                   1
Japan                   1
Mexico                  1
Pakistan                1
Saudi Arabia            1
Singapore               1
South Africa            1
Spain                   1
United Arab Emirates    1
United Kingdom          1
United States           1
Name: Region, dtype: int64

### 2.1 Recover Missing Category and Region Values

Use validated ProductID-to-Category and Country-to-Region relationships to recover missing categorical values.

In [16]:
df_clean["Category"] = (
    df_clean["Category"]
    .str.strip()
    .str.title()
)

In [17]:
category_map = (
    df_clean.dropna(subset=["Category"])
    .drop_duplicates(subset=["ProductID"])
    .set_index("ProductID")["Category"]
)

In [18]:
df_clean["Category"] = (
    df_clean["Category"]
    .fillna(df_clean["ProductID"].map(category_map))
)

In [19]:
region_map = (
    df_clean.dropna(subset=["Region"])
    .drop_duplicates(subset=["Country"])
    .set_index("Country")["Region"]
)

df_clean["Region"] = (
    df_clean["Region"]
    .fillna(df_clean["Country"].map(region_map))
)

In [20]:
df_clean["PaymentMethod"] = (
    df_clean["PaymentMethod"].fillna("Unknown")
)

In [21]:
df_clean.isnull().sum()

OrderID                         0
OrderDate                       0
CustomerID                      0
CustomerRegistrationDate        0
ProductID                       0
ProductName                     0
Category                        0
Quantity                        0
UnitPrice                       0
DiscountPct                     0
GrossSales                      0
DiscountAmount                  0
NetSales                        0
Region                          0
Country                         0
PaymentMethod                   0
ReturnStatus                    0
ReturnDate                  46717
dtype: int64

## 3. Numerical Data Cleaning

Investigate and handle invalid numerical values in Quantity, UnitPrice, and DiscountPct while preserving valid business transactions and maintaining financial calculation consistency.

In [22]:
df_clean[df_clean["UnitPrice"] < 0][
    [
        "OrderID",
        "ProductName",
        "Quantity",
        "UnitPrice",
        "GrossSales",
        "DiscountPct",
        "NetSales",
        "ReturnStatus"
    ]
]

,OrderID,ProductName,Quantity,UnitPrice,GrossSales,DiscountPct,NetSales,ReturnStatus
8800,ORD006501,USB-C Hub,3,-5.0,-15.0,10.0,-13.50,Not Returned
11362,ORD002048,Laptop Sleeve,3,-5.0,-15.0,10.0,-13.50,Not Returned
15345,ORD006957,Running Shoes,1,-5.0,-5.0,10.0,-4.50,Not Returned
16216,ORD016775,Kitchen Organizer,1,-5.0,-5.0,0.0,-5.00,Not Returned
35353,ORD012405,Face Moisturizer,3,-5.0,-15.0,15.0,-12.75,Not Returned
37977,ORD020381,Phone Case,3,-5.0,-15.0,0.0,-15.00,Not Returned
42054,ORD019385,Laptop Sleeve,1,-5.0,-5.0,0.0,-5.00,Not Returned
43108,ORD003642,Laptop Sleeve,3,-5.0,-15.0,0.0,-15.00,Not Returned
44053,ORD006679,Wireless Earbuds,1,-5.0,-5.0,10.0,-4.50,Returned
44621,ORD009782,Water Bottle,4,-5.0,-20.0,0.0,-20.00,Not Returned


In [23]:
df_clean[df_clean["Quantity"] == 0][
    [
        "OrderID",
        "ProductName",
        "Quantity",
        "UnitPrice",
        "GrossSales",
        "ReturnStatus"
    ]
]

,OrderID,ProductName,Quantity,UnitPrice,GrossSales,ReturnStatus
8811,ORD005301,Bluetooth Speaker,0,64.98,0.0,Not Returned
8940,ORD016797,Travel Backpack,0,68.86,0.0,Not Returned
9523,ORD004901,Running Shoes,0,76.38,0.0,Returned
16451,ORD000909,Wireless Earbuds,0,56.34,0.0,Not Returned
21844,ORD016086,Ceramic Mug Set,0,26.31,0.0,Not Returned
22789,ORD017726,Skincare Bundle,0,43.24,0.0,Not Returned
27824,ORD006936,Laptop Sleeve,0,26.97,0.0,Not Returned
29698,ORD018361,Travel Backpack,0,64.12,0.0,Not Returned
32702,ORD011494,Running Shoes,0,67.87,0.0,Not Returned
33374,ORD003288,Kitchen Organizer,0,33.86,0.0,Not Returned


In [24]:
df_clean[df_clean["DiscountPct"] > 100][
    [
        "OrderID",
        "ProductName",
        "Quantity",
        "UnitPrice",
        "DiscountPct",
        "GrossSales",
        "DiscountAmount",
        "NetSales"
    ]
]

,OrderID,ProductName,Quantity,UnitPrice,DiscountPct,GrossSales,DiscountAmount,NetSales
3765,ORD013604,Smart Watch,3,92.70,125.0,278.10,347.62,-69.52
5642,ORD015195,Notebook Set,3,13.23,125.0,39.69,49.61,-9.92
6683,ORD007615,Face Moisturizer,5,25.84,125.0,129.20,161.50,-32.30
7312,ORD000178,Smart Watch,2,105.40,125.0,210.80,263.50,-52.70
11065,ORD018895,Running Shoes,4,73.32,125.0,293.28,366.60,-73.32
14945,ORD000536,Water Bottle,5,20.50,125.0,102.50,128.12,-25.62
37587,ORD005665,Water Bottle,2,20.60,125.0,41.20,51.50,-10.30
37838,ORD015610,Bluetooth Speaker,3,54.70,125.0,164.10,205.12,-41.02
40504,ORD004832,Phone Case,4,16.53,125.0,66.12,82.65,-16.53
43476,ORD003639,Fiction Book,5,13.48,125.0,67.40,84.25,-16.85


### 3.1 Correct Invalid Numerical Values

- Replace negative UnitPrice values using the median valid price for the corresponding product.
- Remove records with zero Quantity.
- Cap DiscountPct values above 100% at 100%.
- Recalculate all derived financial columns to maintain consistency.

In [25]:
product_medians = (
    df_clean.loc[df_clean["UnitPrice"] > 0]
    .groupby("ProductName")["UnitPrice"]
    .median()
)

negative_price_mask = df_clean["UnitPrice"] < 0

df_clean.loc[negative_price_mask, "UnitPrice"] = (
    df_clean.loc[negative_price_mask, "ProductName"]
    .map(product_medians)
)

print("Negative prices remaining:", (df_clean["UnitPrice"] < 0).sum())

Negative prices remaining: 0


In [26]:
# Inspect the median used for Product P001
product_medians.loc[
    df_clean.loc[
        df_clean["ProductID"] == "P001",
        "ProductName"
    ].iloc[0]
]

np.float64(50.045)

In [27]:

# Inspect all records where UnitPrice was originally negative
# and replaced using the product-level median.

imputed_prices = df_clean.loc[
    negative_price_mask,
    [
        "OrderID",
        "ProductID",
        "ProductName",
        "Quantity",
        "UnitPrice",
    ]
].copy()

# Compare the imputed prices with their two-decimal rounded values.
imputed_prices["RoundedUnitPrice"] = (
    imputed_prices["UnitPrice"].round(2)
)

# Calculate the rounding difference.
imputed_prices["RoundingDifference"] = (
    imputed_prices["UnitPrice"]
    - imputed_prices["RoundedUnitPrice"]
)

# Display the imputed records.
display(imputed_prices)

# Summarise the number of imputed prices affected by rounding.
print(
    "Imputed prices with rounding differences:",
    (imputed_prices["RoundingDifference"].abs() > 0).sum()
)


,OrderID,ProductID,ProductName,Quantity,UnitPrice,RoundedUnitPrice,RoundingDifference
8800,ORD006501,P004,USB-C Hub,3,30.050,30.05,0.000
11362,ORD002048,P006,Laptop Sleeve,3,25.010,25.01,0.000
15345,ORD006957,P008,Running Shoes,1,75.200,75.20,0.000
16216,ORD016775,P012,Kitchen Organizer,1,35.020,35.02,0.000
35353,ORD012405,P016,Face Moisturizer,3,23.020,23.02,0.000
37977,ORD020381,P005,Phone Case,3,15.020,15.02,0.000
42054,ORD019385,P006,Laptop Sleeve,1,25.010,25.01,0.000
43108,ORD003642,P006,Laptop Sleeve,3,25.010,25.01,0.000
44053,ORD006679,P001,Wireless Earbuds,1,50.045,50.04,0.005
44621,ORD009782,P015,Water Bottle,4,18.950,18.95,0.000


Imputed prices with rounding differences: 1


In [28]:

# Standardise UnitPrice to two decimal places
# before recalculating the financial columns.

df_clean["UnitPrice"] = df_clean["UnitPrice"].round(2)

print(
    "UnitPrice values with more than two decimal places:",
    (
        df_clean["UnitPrice"] != df_clean["UnitPrice"].round(2)
    ).sum()
)

# Verify the corrected price for the affected record.
display(
    df_clean.loc[
        df_clean["OrderID"] == "ORD006679",
        ["OrderID", "ProductID", "Quantity", "UnitPrice"]
    ]
)


UnitPrice values with more than two decimal places: 0


,OrderID,ProductID,Quantity,UnitPrice
28658,ORD006679,P016,5,21.62
44053,ORD006679,P001,1,50.04


In [29]:
zero_quantity_count = (df_clean["Quantity"] == 0).sum()

df_clean = df_clean[df_clean["Quantity"] > 0].copy()

print("Zero-quantity records removed:", zero_quantity_count)
print("Remaining zero quantities:", (df_clean["Quantity"] == 0).sum())

Zero-quantity records removed: 15
Remaining zero quantities: 0


In [30]:
invalid_discount_count = (df_clean["DiscountPct"] > 100).sum()

df_clean["DiscountPct"] = df_clean["DiscountPct"].clip(upper=100)

print("Invalid discounts corrected:", invalid_discount_count)
print("Discounts above 100% remaining:", (df_clean["DiscountPct"] > 100).sum())

Invalid discounts corrected: 12
Discounts above 100% remaining: 0


In [31]:
df_clean["GrossSales"] = (
    df_clean["Quantity"] * df_clean["UnitPrice"]
).round(2)

df_clean["DiscountAmount"] = (
    df_clean["GrossSales"] * df_clean["DiscountPct"] / 100
).round(2)

df_clean["NetSales"] = (
    df_clean["GrossSales"] - df_clean["DiscountAmount"]
).round(2)

In [32]:
print("Negative prices:", (df_clean["UnitPrice"] < 0).sum())
print("Zero quantities:", (df_clean["Quantity"] == 0).sum())
print("Discounts above 100%:", (df_clean["DiscountPct"] > 100).sum())
print("Remaining duplicate records:", df_clean.duplicated().sum())

Negative prices: 0
Zero quantities: 0
Discounts above 100%: 0
Remaining duplicate records: 0


In [33]:
print(
    "Maximum categories per product:",
    df_clean.groupby("ProductID")["Category"].nunique().max()
)

print(
    "Maximum regions per country:",
    df_clean.groupby("Country")["Region"].nunique().max()
)

print("\nRemaining missing values:")
print(df_clean.isnull().sum())

Maximum categories per product: 1
Maximum regions per country: 1

Remaining missing values:
OrderID                         0
OrderDate                       0
CustomerID                      0
CustomerRegistrationDate        0
ProductID                       0
ProductName                     0
Category                        0
Quantity                        0
UnitPrice                       0
DiscountPct                     0
GrossSales                      0
DiscountAmount                  0
NetSales                        0
Region                          0
Country                         0
PaymentMethod                   0
ReturnStatus                    0
ReturnDate                  46703
dtype: int64


## 4. Date and Data Type Standardisation

In [34]:
date_columns = [
    "OrderDate",
    "CustomerRegistrationDate",
    "ReturnDate"
]

for col in date_columns:
    df_clean[col] = pd.to_datetime(df_clean[col], errors="coerce")

df_clean[date_columns].dtypes

OrderDate                   datetime64[ns]
CustomerRegistrationDate    datetime64[ns]
ReturnDate                  datetime64[ns]
dtype: object

In [35]:
df_clean.dtypes

OrderID                             object
OrderDate                   datetime64[ns]
CustomerID                          object
CustomerRegistrationDate    datetime64[ns]
ProductID                           object
ProductName                         object
Category                            object
Quantity                             int64
UnitPrice                          float64
DiscountPct                        float64
GrossSales                         float64
DiscountAmount                     float64
NetSales                           float64
Region                              object
Country                             object
PaymentMethod                       object
ReturnStatus                        object
ReturnDate                  datetime64[ns]
dtype: object

In [36]:
print("Missing OrderDate:", df_clean["OrderDate"].isna().sum())
print(
    "Missing CustomerRegistrationDate:",
    df_clean["CustomerRegistrationDate"].isna().sum()
)
print("Missing ReturnDate:", df_clean["ReturnDate"].isna().sum())

Missing OrderDate: 0
Missing CustomerRegistrationDate: 0
Missing ReturnDate: 46703


In [37]:
print(
    "DiscountAmount mismatches:",
    ((df_clean["DiscountAmount"] -
      df_clean["GrossSales"] * df_clean["DiscountPct"] / 100).abs() > 0.01).sum()
)

print(
    "NetSales mismatches:",
    ((df_clean["NetSales"] -
      (df_clean["GrossSales"] - df_clean["DiscountAmount"])).abs() > 0.01).sum()
)

DiscountAmount mismatches: 0
NetSales mismatches: 0


## 5. Final Data Quality Validation

In [38]:
print("Final dataset shape:", df_clean.shape)
print("Duplicate records:", df_clean.duplicated().sum())

print("\nMissing values:")
print(df_clean.isnull().sum())

print("\nNumerical validation:")
print("Negative prices:", (df_clean["UnitPrice"] < 0).sum())
print("Zero quantities:", (df_clean["Quantity"] == 0).sum())
print("Invalid discounts:", (df_clean["DiscountPct"] > 100).sum())

print("\nFinancial validation:")
print(
    "GrossSales mismatches:",
    ((df_clean["GrossSales"] -
      df_clean["Quantity"] * df_clean["UnitPrice"]).abs() > 0.01).sum()
)

Final dataset shape: (49885, 18)
Duplicate records: 0

Missing values:
OrderID                         0
OrderDate                       0
CustomerID                      0
CustomerRegistrationDate        0
ProductID                       0
ProductName                     0
Category                        0
Quantity                        0
UnitPrice                       0
DiscountPct                     0
GrossSales                      0
DiscountAmount                  0
NetSales                        0
Region                          0
Country                         0
PaymentMethod                   0
ReturnStatus                    0
ReturnDate                  46703
dtype: int64

Numerical validation:
Negative prices: 0
Zero quantities: 0
Invalid discounts: 0

Financial validation:
GrossSales mismatches: 0


In [39]:
df_clean.to_csv(
    "../Dataset/Cleaned/ecommerce_sales_customer_cleaned.csv",
    index=False
)

print("Cleaned dataset saved successfully.")

Cleaned dataset saved successfully.
